# D73 续冲 AB(死锁手术后)——`ppo_d73_AB_s0`

**全新 run-tag, 不覆盖旧文件**。init/锚 = D72 push_AB 的 best(157,245), fv=2(维度已匹配, 无需零填充)。与 A 版保持等训练量, 供「前瞻特征中性」消融的公平对照。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 训练(全新 run-tag, **不覆盖旧文件**; 重跑本格=续训自己)

In [ ]:
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag ppo_d73_AB_s0 --feature-version 2 \
    --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt \
    --bc-anchor-beta 0.01 --lr 5e-5 \
    --encoder gnn --control-classes 12 \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 \
    --promote-at 1.0 --promote-delay-k 1.0 \
    --train-eval-days 6 --iters 100 --steps-per-iter 8192 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume


## 对比(新旧四个 run + 基线)

In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): return json.loads((rep/f'{tag}.json').read_text())
    if (rep/f'{tag}_last.pt').exists(): return torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    return None
runs=[('D73 A','ppo_d73_A_s0'),('D73 AB','ppo_d73_AB_s0'),('D72 A(修死锁前)','ppo_push_A_s0'),('D72 AB','ppo_push_AB_s0')]
loaded={n:load(t) for n,t in runs}
base=next((d.get('baselines') for d in loaded.values() if d and d.get('baselines')),{})
print(f"{'run':<16}{'best完成':>9}{'best延误':>11}{'best_rew':>10}{'@it':>5}")
for k in ['fcfs','lookup_most_frequent','greedy_set','signaller_replay']:
    if k in base:
        b=base[k]; print(f"{k:<16}{b['eval_completion_rate']:>9.3f}{b['eval_weighted_secondary_delay_s']:>11.0f}{b['eval_episode_reward']:>10.0f}")
for name,_ in runs:
    d=loaded.get(name)
    if not d or not d.get('best'): print(f'{name:<16}(missing)'); continue
    b=d['best']; print(f"{name:<16}{b.get('completion',0):>9.3f}{b.get('delay',0):>11.0f}{b.get('reward',0):>10.0f}{b.get('iter',-1):>5}")
# 升档轨迹(验证档3不再卡死)
for name in ['D73 A','D73 AB']:
    d=loaded.get(name)
    if d and d.get('evals'):
        tiers=[(e['iter'],e.get('tier',0)+1,e.get('train_completion'),e.get('train_weighted_secondary_delay_s')) for e in d['evals']]
        print(f'\n{name} 升档轨迹(it,档,train完成,train延误):')
        for row in tiers: print('  ',row)
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    plt.figure(figsize=(8,3.2))
    for name,_ in runs:
        d=loaded.get(name)
        if d and d.get('evals'):
            ev=d['evals']; plt.plot([e['iter'] for e in ev],[e['eval_weighted_secondary_delay_s'] for e in ev],marker='.',label=name)
    if 'fcfs' in base: plt.axhline(base['fcfs']['eval_weighted_secondary_delay_s'],ls='--',c='b',lw=.9,label='fcfs 181k')
    plt.ylim(130000, 260000); plt.ylabel('val 延误'); plt.xlabel('iter'); plt.legend(fontsize=8)
    plt.title('D73 死锁手术后续训'); plt.tight_layout(); plt.savefig('artifacts/reports/d73_push.png',dpi=110)
    print('fig -> artifacts/reports/d73_push.png')
except Exception as e: print('plot skip',e)
